## 🏋️ Exercise: Meeting notes → action items

Build a little tool that turns messy meeting notes into a saved to-do list.

1. Reuse the `meeting.txt` file from Part 3 (or write your own).
2. Read it, and ask the LLM to **extract the action items** — who needs to do what.
3. Save the result to `action_items.txt`.

**Bonus (ties everything together):** instead of plain text, ask for **structured output** (a Pydantic model with a list of `{task, owner}`), then save it with `json.dump` to `action_items.json`. Load it back and print each task. Now you've combined files + structured output into one mini-app. 🚀

In [16]:
with open("meeting.txt", "r", encoding="utf-8") as f:
    meeting_notes = f.read()

print(meeting_notes)

Project Meeting Notes

- Rahul will finish the login page by Friday.
- Priya needs to test the API and report any bugs by Monday.
- We need to update the project documentation. Amit will handle this.
- The team agreed to fix the database connection issue. Rahul will investigate it.
- Priya will prepare the demo presentation before the next meeting.


In [17]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

In [19]:
from pydantic import BaseModel
from google.genai import types
import json

# 1. Define the structure of each action item
class ActionItem(BaseModel):
    task: str
    owner: str

# 2. Define the complete output structure
class ActionItems(BaseModel):
    action_items: list[ActionItem]

# 3. Read meeting notes from the file
with open("meeting.txt", "r", encoding="utf-8") as f:
    meeting_notes=f.read()

# 4. Ask Gemini to extract the action items
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=f"""
    Extract all action items from these meeting notes.

    For each item:
    - task: describe what needs to be done
    - owner: the person's name, or "Unassigned" if unknown

    Do not invent owners or tasks.

    Meeting notes:
    {meeting_notes}
    """,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=ActionItems
    )
)

# 5. Parse the structured response
result = response.parsed

# 6. Save the result to a JSON file
with open("action_items.json", "w", encoding="utf-8") as f:
    json.dump(
        result.model_dump(),
        f,
        indent=4,
        ensure_ascii=False
    )

print("Actions items saved successfully!")

Actions items saved successfully!


In [20]:
import json

# Step 1: Load the saved JSON file
with open("action_items.json", "r", encoding="utf-8") as f:
    saved_data = json.load(f)

# Step 2: Print each action item
for item in saved_data["action_items"]:
    print(f"Task: {item['task']}")
    print(f"Owner: {item['owner']}")
    print("-" * 40)

# Step 3: Save action items to a text file
with open("action_items.txt", "w", encoding="utf-8") as f:
    for item in saved_data["action_items"]:
        f.write(f"Task: {item['task']}\n")
        f.write(f"Owner: {item['owner']}\n")
        f.write("\n")

print("\nAction items saved successfully to action_items.txt!")

Task: Finish the login page by Friday
Owner: Rahul
----------------------------------------
Task: Test the API and report any bugs by Monday
Owner: Priya
----------------------------------------
Task: Update the project documentation
Owner: Amit
----------------------------------------
Task: Investigate the database connection issue
Owner: Rahul
----------------------------------------
Task: Prepare the demo presentation before the next meeting
Owner: Priya
----------------------------------------

Action items saved successfully to action_items.txt!
